# Dataset Download

Downloads all datasets to `data/raw/` for all three language groups (C/C++, Java, Python).

**Run once** before any ingestion notebook. Each section is idempotent (skips if file already present).

| # | Dataset | Source | Format | Notes |
|---|---------|--------|--------|-------|
| 1 | PrimeVul | HuggingFace starsofchance/PrimeVul | JSONL | |
| 2 | ICVul | Google Drive | CSV | |
| 3 | CVEfixes | HuggingFace hitoshura25/cvefixes | Parquet | C+Java+Python |
| 4 | MegaVul | HuggingFace hitoshura25/megavul | Parquet | |
| 5 | SecVulEval | HuggingFace arag0rn/SecVulEval | CSV | |
| 6 | CrossVul | Zenodo (crossvul.zip) | ZIP | C+Java+Python |
| 7 | SVEN | HuggingFace bstee615/sven | Parquet | C+Python |
| 8 | Juliet C/C++ | NIST SARD (auto) | ZIP | |
| 9 | Juliet Java | NIST SARD (auto) | ZIP | |
| 10 | CASTLE | GitHub CASTLE-Benchmark | JSON | |
| 11 | LLMSecEval | Zenodo 5225651 + GitHub tuhh-softsec/LLMSecEval | dir | C/C++ + Python |
| 12 | OWASP Benchmark | GitHub OWASP-Benchmark | ZIP | Java+Python |
| 13 | CAPEC_LLM | GitHub llmForCapec/CAPECDatasetsLLM | JSON | Java+Python |
| 14 | PatchEval | GitHub bytedance/PatchEval | JSON | Python |
| 15 | PyVul | GitHub billquan/PyVul | JSON/JSONL | Python |
| 16 | SecurityEval | GitHub s2e-lab/SecurityEval | JSONL | Python |

In [39]:
import subprocess
import shutil
import urllib.request
import zipfile
import tarfile
from pathlib import Path

RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)
print(f'Data directory: {RAW.resolve()}')

def _git_clone(url: str, dest: Path, depth: int = 1):
    """Shallow-clone a GitHub repo; skip if dest already exists."""
    if dest.exists() and any(dest.iterdir()):
        print(f'  {dest.name}: already present -- skipping.')
        return
    dest.mkdir(parents=True, exist_ok=True)
    print(f'  Cloning {url} ...')
    result = subprocess.run(
        ['git', 'clone', f'--depth={depth}', url, str(dest)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr.strip()}')
    else:
        files = sum(1 for _ in dest.rglob('*') if _.is_file())
        print(f'  {dest.name}: cloned ({files} files)')

def _hf_snapshot(repo_id: str, dest: Path, repo_type: str = 'dataset'):
    """Download full HuggingFace repo snapshot; skip if dest has files."""
    if dest.exists() and any(dest.iterdir()):
        print(f'  {dest.name}: already present -- skipping.')
        return
    from huggingface_hub import snapshot_download
    print(f'  Downloading {repo_id} ...')
    local = snapshot_download(repo_id=repo_id, repo_type=repo_type)
    shutil.copytree(local, dest, dirs_exist_ok=True)
    files = sum(1 for _ in dest.rglob('*') if _.is_file())
    print(f'  {dest.name}: saved ({files} files)')

Data directory: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\data\raw


## 1. PrimeVul

HuggingFace `starsofchance/PrimeVul` -- train and test JSONL splits.

In [40]:
primevul_train = RAW / 'primevul_train.jsonl'
primevul_test  = RAW / 'primevul_test.jsonl'

if primevul_train.exists() and primevul_test.exists():
    print('PrimeVul already downloaded -- skipping.')
else:
    from huggingface_hub import hf_hub_download
    for split, out_path in [('primevul_train.jsonl', primevul_train),
                             ('primevul_test.jsonl',  primevul_test)]:
        print(f'Downloading {split} ...')
        local = hf_hub_download(repo_id='starsofchance/PrimeVul', filename=split, repo_type='dataset')
        shutil.copy(local, out_path)
        print(f'  Saved {out_path.name} ({out_path.stat().st_size / 1e6:.1f} MB)')

PrimeVul already downloaded -- skipping.


## 2. ICVul

Google Drive shared archive. Requires `gdown`.

In [41]:
import gdown

icvul_dir = RAW / 'icvul'
ICVUL_FILE_ID = '1Bnnb7kJa8GEfyESIAuGXj2z0g8FvXgRk'

if icvul_dir.exists() and any(icvul_dir.iterdir()):
    print('ICVul already downloaded -- skipping.')
else:
    icvul_dir.mkdir(parents=True, exist_ok=True)
    archive_path = RAW / '_icvul_archive'
    print('Downloading ICVul from Google Drive ...')
    downloaded = gdown.download(id=ICVUL_FILE_ID, output=str(archive_path))
    if downloaded is None:
        raise RuntimeError('gdown failed -- check file permissions')
    candidates = list(RAW.glob('_icvul_archive*'))
    archive_path = candidates[0]
    print(f'Downloaded: {archive_path.name} ({archive_path.stat().st_size / 1e6:.1f} MB)')
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(icvul_dir)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            tf.extractall(icvul_dir)
    archive_path.unlink(missing_ok=True)
    csv_files = list(icvul_dir.rglob('*.csv'))
    print(f'ICVul: {len(csv_files)} CSV files in {icvul_dir}')

ICVul already downloaded -- skipping.


## 3. CVEfixes

HuggingFace `hitoshura25/cvefixes` -- 3 Parquet shards. Covers C, Java, Python.

In [42]:
from huggingface_hub import hf_hub_download

cvefixes_dir = RAW / 'cvefixes'
CVEFIXES_FILES = ['data/train-00000-of-00003.parquet', 'data/train-00001-of-00003.parquet', 'data/train-00002-of-00003.parquet']

if cvefixes_dir.exists() and len(list(cvefixes_dir.glob('*.parquet'))) == len(CVEFIXES_FILES):
    print('CVEfixes already downloaded -- skipping.')
else:
    cvefixes_dir.mkdir(parents=True, exist_ok=True)
    for hf_path in CVEFIXES_FILES:
        local = hf_hub_download(repo_id='hitoshura25/cvefixes', filename=hf_path, repo_type='dataset')
        dest = cvefixes_dir / Path(hf_path).name
        shutil.copy(local, dest)
        print(f'  {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)')
    print(f'CVEfixes saved to {cvefixes_dir}')

CVEfixes already downloaded -- skipping.


## 4. MegaVul

HuggingFace `hitoshura25/megavul` -- 2 Parquet shards.

In [43]:
megavul_dir = RAW / 'megavul'
MEGAVUL_FILES = ['data/train-00000-of-00002.parquet', 'data/train-00001-of-00002.parquet']

if megavul_dir.exists() and len(list(megavul_dir.glob('*.parquet'))) == len(MEGAVUL_FILES):
    print('MegaVul already downloaded -- skipping.')
else:
    megavul_dir.mkdir(parents=True, exist_ok=True)
    for hf_path in MEGAVUL_FILES:
        local = hf_hub_download(repo_id='hitoshura25/megavul', filename=hf_path, repo_type='dataset')
        dest = megavul_dir / Path(hf_path).name
        shutil.copy(local, dest)
        print(f'  {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)')
    print(f'MegaVul saved to {megavul_dir}')

MegaVul already downloaded -- skipping.


## 5. SecVulEval

HuggingFace `arag0rn/SecVulEval`.

In [44]:
secvuleval_path = RAW / 'secvuleval.csv'

if secvuleval_path.exists():
    print('SecVulEval already downloaded -- skipping.')
else:
    import pandas as pd
    from datasets import load_dataset
    ds = load_dataset('arag0rn/SecVulEval', trust_remote_code=True)
    df = pd.concat([ds[split].to_pandas() for split in ds], ignore_index=True)
    df.to_csv(secvuleval_path, index=False)
    print(f'SecVulEval: {len(df)} rows -> {secvuleval_path}')

SecVulEval already downloaded -- skipping.


## 6. CrossVul

Zenodo `crossvul.zip` — Covers C/C++, Java, Python. Structure: `dataset_final_sorted/CWE-NNN/<lang>/bad_*|good_*` (no extensions; `bad_`=vulnerable, `good_`=safe).

In [45]:
crossvul_zip = RAW / 'crossvul.zip'
if crossvul_zip.exists():
    print(f'CrossVul already present ({crossvul_zip.stat().st_size / 1e6:.0f} MB) -- skipping.')
else:
    print(f'crossvul.zip not found at {crossvul_zip}')
    print('Download manually from Zenodo and place it there.')

CrossVul already present (367 MB) -- skipping.


## 7. SVEN

HuggingFace `bstee615/sven` -- Parquet. Covers C/C++, Python.

In [46]:
sven_dir = RAW / 'sven'

if sven_dir.exists() and any(sven_dir.glob('*.parquet')):
    print('SVEN already downloaded -- skipping.')
else:
    sven_dir.mkdir(parents=True, exist_ok=True)
    print('Downloading SVEN (bstee615/sven) ...')
    _hf_snapshot('bstee615/sven', sven_dir)
    parquets = list(sven_dir.rglob('*.parquet'))
    print(f'SVEN: {len(parquets)} parquet file(s)')

  sven: already present -- skipping.
SVEN: 2 parquet file(s)


## 8. Juliet C/C++

NIST SARD Juliet Test Suite for C/C++ v1.3 -- 64,099 test cases across 118 CWEs (~146 MB).

In [47]:
JULIET_C_URL = 'https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3.zip'
juliet_c = RAW / 'juliet_c.zip'

if juliet_c.exists():
    print(f'Juliet C/C++ present ({juliet_c.stat().st_size / 1e6:.0f} MB).')
else:
    print('Downloading Juliet C/C++ (~146 MB) ...')
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')]
    urllib.request.install_opener(opener)
    
    try:
        urllib.request.urlretrieve(JULIET_C_URL, juliet_c)
        print(f'Juliet C/C++ saved ({juliet_c.stat().st_size / 1e6:.0f} MB).')
    except Exception as e:
        print(f"Error during download: {e}")

Juliet C/C++ present (153 MB).


## 9. Juliet Java

NIST SARD Juliet Test Suite for Java v1.3 -- 28,881 test cases across 112 CWEs (~73 MB).

In [48]:
JULIET_JAVA_URL = 'https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-java-v1-3.zip'
juliet_java = RAW / 'juliet_java.zip'

if juliet_java.exists():
    print(f'Juliet Java present ({juliet_java.stat().st_size / 1e6:.0f} MB).')
else:
    print('Downloading Juliet Java (~73 MB) ...')
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')]
    urllib.request.install_opener(opener)

    try:
        urllib.request.urlretrieve(JULIET_JAVA_URL, juliet_java)
        print(f'Juliet Java saved ({juliet_java.stat().st_size / 1e6:.0f} MB).')
    except Exception as e:
        print(f"Error during download: {e}")

Juliet Java present (77 MB).


## 10. CASTLE

GitHub `CASTLE-Benchmark/CASTLE-Benchmark` -- JSON benchmark file.

In [49]:
castle_dir = RAW / 'castle'
_git_clone('https://github.com/CASTLE-Benchmark/CASTLE-Benchmark.git', castle_dir)

  castle: already present -- skipping.


## 11. LLMSecEval

Two-part download:
- **Vulnerable (C/C++ + Python)**: Zenodo record 5225651 (`copilot-cwe-scenarios-dataset`) — Copilot-generated code extracted to `llmseceval/zenodo/`
- **Safe (Python only)**: GitHub `tuhh-softsec/LLMSecEval` — `Dataset/Secure Code Samples/` copied to `llmseceval/CWE-NNN/Secure/`

In [50]:
import re

llmseceval_dir = RAW / 'llmseceval'
COPILOT_ZIP    = RAW / 'copilot-cwe-scenarios-dataset.zip'
GITHUB_URL     = 'https://github.com/tuhh-softsec/LLMSecEval.git'

_zenodo_dir     = llmseceval_dir / 'zenodo'
_zenodo_present = _zenodo_dir.exists() and any(_zenodo_dir.rglob('gen_scenario/*.py'))
_secure_present = llmseceval_dir.exists() and any(llmseceval_dir.glob('CWE-*/Secure/*.py'))

if _zenodo_present and _secure_present:
    py_v = sum(1 for _ in _zenodo_dir.rglob('gen_scenario/*.py'))
    c_v  = sum(1 for _ in _zenodo_dir.rglob('gen_scenario/*.c'))
    py_s = sum(1 for _ in llmseceval_dir.glob('CWE-*/Secure/*.py'))
    print(f'LLMSecEval already present -- {py_v} Python vuln, {c_v} C vuln, {py_s} Python safe.')
else:
    llmseceval_dir.mkdir(parents=True, exist_ok=True)

    # -- Vulnerable code: extract from local ZIP --
    if not _zenodo_present:
        if not COPILOT_ZIP.exists():
            print(f'  copilot-cwe-scenarios-dataset.zip not found at {COPILOT_ZIP}')
            print('  Download from https://zenodo.org/records/5225651 and place it there.')
        else:
            print(f'Extracting {COPILOT_ZIP.name} ({COPILOT_ZIP.stat().st_size / 1e6:.1f} MB) ...')
            _zenodo_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(COPILOT_ZIP) as zf:
                zf.extractall(_zenodo_dir)
            py_v = sum(1 for _ in _zenodo_dir.rglob('gen_scenario/*.py'))
            c_v  = sum(1 for _ in _zenodo_dir.rglob('gen_scenario/*.c'))
            print(f'  Extracted: {py_v} Python vuln, {c_v} C vuln files')

    # -- Safe Python code from GitHub --
    if not _secure_present:
        github_clone = RAW / '_llmseceval_github'
        _git_clone(GITHUB_URL, github_clone)
        secure_src = github_clone / 'Dataset' / 'Secure Code Samples'
        n_copied = 0
        if secure_src.exists():
            for cwe_subdir in secure_src.iterdir():
                if not cwe_subdir.is_dir():
                    continue
                m = re.search(r'\d+', cwe_subdir.name)
                if not m:
                    continue
                dest_secure = llmseceval_dir / f'CWE-{int(m.group())}' / 'Secure'
                dest_secure.mkdir(parents=True, exist_ok=True)
                for f in cwe_subdir.iterdir():
                    if f.is_file():
                        shutil.copy(f, dest_secure / f.name)
                        n_copied += 1
        shutil.rmtree(github_clone, ignore_errors=True)
        print(f'  Copied {n_copied} safe Python files from GitHub')

Extracting copilot-cwe-scenarios-dataset.zip (2.4 MB) ...


  Extracted: 571 Python vuln, 516 C vuln files
  Cloning https://github.com/tuhh-softsec/LLMSecEval.git ...
  _llmseceval_github: cloned (923 files)
  Copied 146 safe Python files from GitHub


## 12. OWASP Benchmark

GitHub `OWASP-Benchmark/BenchmarkJava` (Java) and `OWASP-Benchmark/BenchmarkPython` (Python).

In [51]:
owasp_java_dir   = RAW / 'owasp_benchmark'
owasp_python_dir = RAW / 'owasp_benchmark_python'

_git_clone('https://github.com/OWASP-Benchmark/BenchmarkJava.git',   owasp_java_dir)
_git_clone('https://github.com/OWASP-Benchmark/BenchmarkPython.git', owasp_python_dir)

  owasp_benchmark: already present -- skipping.
  owasp_benchmark_python: already present -- skipping.


## 13. CAPEC_LLM

GitHub `llmForCapec/CAPECDatasetsLLM` -- LLM-generated code snippets for CAPEC entries, covering Java and Python.

In [52]:
capec_dir = RAW / 'capec_llm'
_git_clone('https://github.com/llmForCapec/CAPECDatasetsLLM.git', capec_dir)

  capec_llm: already present -- skipping.


## 14. PatchEval

GitHub `bytedance/PatchEval` -- Python vulnerability/fix pairs from CVEs.

In [53]:
patcheval_dir = RAW / 'patcheval'
_git_clone('https://github.com/bytedance/PatchEval.git', patcheval_dir)

  patcheval: already present -- skipping.


## 15. PyVul

GitHub `billquan/PyVul` -- Python vulnerability dataset.

In [54]:
pyvul_dir = RAW / 'pyvul'
_git_clone('https://github.com/billquan/PyVul.git', pyvul_dir)

  pyvul: already present -- skipping.


## 16. SecurityEval

GitHub `s2e-lab/SecurityEval` -- LLM-generated insecure Python code, JSONL format.

In [56]:
security_eval_dir = RAW / 'security_eval'
_git_clone('https://github.com/s2e-lab/SecurityEval.git', security_eval_dir)

  security_eval: already present -- skipping.


## 18. Verify all downloads

Confirms all expected paths exist.

In [ ]:
import pandas as pd

# (path, display_name, required)
CHECKS = [
    (RAW / 'primevul_train.jsonl',   'PrimeVul train',    True),
    (RAW / 'primevul_test.jsonl',    'PrimeVul test',     True),
    (RAW / 'icvul',                  'ICVul',             True),
    (RAW / 'cvefixes',               'CVEfixes',          True),
    (RAW / 'megavul',                'MegaVul',           True),
    (RAW / 'secvuleval.csv',         'SecVulEval',        True),
    (RAW / 'crossvul.zip',           'CrossVul',          True),
    (RAW / 'sven',                   'SVEN',              True),
    (RAW / 'juliet_c.zip',           'Juliet C/C++',      True),
    (RAW / 'juliet_java.zip',        'Juliet Java',       True),
    (RAW / 'castle',                 'CASTLE',            True),
    (RAW / 'llmseceval',             'LLMSecEval',        True),
    (RAW / 'owasp_benchmark',        'OWASP Java',        True),
    (RAW / 'owasp_benchmark_python', 'OWASP Python',      True),
    (RAW / 'capec_llm',              'CAPEC_LLM',         True),
    (RAW / 'patcheval',              'PatchEval',         True),
    (RAW / 'pyvul',                  'PyVul',             True),
    (RAW / 'security_eval',          'SecurityEval',      True),
]

missing_required = []
for path, name, required in CHECKS:
    if path.exists():
        if path.is_file():
            info = f'{path.stat().st_size / 1e6:.1f} MB'
        else:
            n = sum(1 for _ in path.rglob('*') if _.is_file())
            info = f'{n} files'
        print(f'OK   {name:<26} {info}')
    else:
        print(f'--   {name:<26} MISSING')
        if required:
            missing_required.append(name)

print()
if missing_required:
    print(f'Required datasets missing: {missing_required}')
    print('Re-run the relevant section above.')
else:
    print('All required datasets present.')